# Mini Project 1 — Analysis Notebook

**Your name:**  Ruofu Li
**Dataset:**  Riot Developer API
**Date:**  May 13, 2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [50]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

!pip install plotly kaleido
fig.write_image("chart_name.png")

print("Setup complete.")


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Setup complete.


---

## Section 1 — Overview

**Dataset:** A combination of 3 different APIs from the Riot Developer API, including Summoner ID, Match by PUUID, and Match Details by Match ID. 

**Why this dataset:** I play League of Legends and I was curious about the champion picking habits of other players around my skill level. 

**Three analytical questions:**

1. Who is the most played Support champion in Bronze IV Solo 5x5 Ranked lobbies? 
2. What is the win rate for the top 3 Support champions in this lobby? 
3. Which Support champion has the highest win rate? 

**If time allows:** 

4. What is the distribution for the most played Support champion across all Level IV ranks (ex: Silver IV, Platinum IV, etc.)? 
5. What is the most common Bottom Lane/Support pairing in Bronze-level lobbies? What about across all ranks? 

*Some of these questions have been adjusted since the initial project proposal 

**What a practitioner would do with these findings:** Probably learn how to play other champions that best counter the most popular Support picks, or at least attempt to get better at them. 

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

**Note** 
Please refer to prep work to create `support_picks.csv` in `a6_preparation_notebook.ipynb`, which was copy and pasted from A5.

In [58]:
from pathlib import Path

data_path = Path("..") / "Week 5" / "support_picks.csv"
df = pd.read_csv(data_path)

# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 996 entries, 0 to 995
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   match_id            996 non-null    str  
 1   puuid               996 non-null    str  
 2   championId          996 non-null    int64
 3   championName        996 non-null    str  
 4   teamId              996 non-null    int64
 5   teamPosition        996 non-null    str  
 6   individualPosition  996 non-null    str  
 7   win                 996 non-null    bool 
dtypes: bool(1), int64(2), str(5)
memory usage: 55.6 KB


In [55]:
df.head()

,match_id,puuid,championId,championName,teamId,teamPosition,individualPosition,win
0,NA1_5550304472,UL3qXsy63hmThQw39LTtPWK2JI316v5jHX8r_yw5g9A873...,235,Senna,100,UTILITY,UTILITY,False
1,NA1_5550304472,o9UXKEi4u2kwH6F9UJVxFy8x6Jbi2tXl3Y5sFmPaDBR8Us...,89,Leona,200,UTILITY,UTILITY,True
2,NA1_5545162641,iAPisvb-m-cDkpVmzVJtNKdNjTwoIgk-PkKIvRQScvPeed...,53,Blitzcrank,100,UTILITY,UTILITY,True
3,NA1_5545162641,z47LTXaQxM72C5dDdLwpdhuZJVDjcmnqkjE11WKKhqtDPx...,26,Zilean,200,UTILITY,UTILITY,False
4,NA1_5530609519,Kj3raKvthe46haxKC6N2LOpUK8_VxgOyRUo48bufObv60i...,22,Ashe,100,UTILITY,UTILITY,True


In [56]:
# Summary statistics for numeric columns
df.describe()

#included from the template provided, but inapplicable to this dataset because it relies on string data instead of integers 

,championId,teamId
count,996.000000,996.000000
mean,204.387550,150.100402
std,222.639688,50.025018
min,1.000000,100.000000
25%,53.000000,100.000000
50%,111.000000,200.000000
75%,267.000000,200.000000
max,910.000000,200.000000


In [19]:
# Percentage of wins (True) vs. losses (False)
win_loss_pct = (
    df["win"]
    .value_counts(normalize=True)
    .mul(100)
    .rename(index={True: "wins", False: "losses"})
)
win_loss_pct

win
wins      50.100402
losses    49.899598
Name: proportion, dtype: float64

**Your data profile notes**  

To answer the questions above: 
- This dataset has 8 columns and 996 rows, including a row for column titles. 
- Each column represents a different characteristic of each champion that was picked, including Match ID, PUUIDs for each player, champion ID associated with each champion, the champion's name, which team each player is on, team position, individual position, and if that player was part of the winning team. 
- No inconsistencies were identified, but I was a little confused on what the difference was between Individual Position and Team Position. They are all the same for each champion in the dataset (Utility for both)
- I'll be focusing on champion name, win, and probably team ID for my analysis. These fields will allow me to answer my questions concerning which Support champion is the most popular pick, win rates associated with these champions, and team composition the best. 

Other observations: 
- I included the code cell that took summary statistics for the DataFrame, but it was pretty irrelevant because the most significant fields for my analytical questions are mostly strings. 
- Even calculating the win rates isn't very helpful because of how the dataset was organized; there will always be one winning Support and one losing Support for each Match ID, so it makes sense that the Wins/Losses are about 50/50. 

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** Who is the most played Support champion in Bronze IV Solo 5x5 Ranked lobbies? 


In [24]:
# Count how often each support champion appears, then sort most → least
champion_pick_counts = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"))
    .sort_values("pick_count", ascending=False)
)

champion_pick_counts.head(10)

,championName,pick_count
40,Lux,75
47,Morgana,66
62,Seraphine,61
83,Yuumi,48
61,Senna,38
35,Leona,34
67,Sona,34
55,Pyke,32
49,Nautilus,32
74,Thresh,30


The most played Support champion in Bronze IV Solo 5x5 Ranked lobbies is Lux, followed by Morgana and Seraphine. It's important to note that this sample size is far too small to be truly statistically significant, but it's a nice start to start establishing early patterns. 

**Interpretation:**  
The most played Support champion in Bronze IV Solo 5x5 Ranked lobbies is Lux, followed by Morgana and Seraphine. It's important to note that this sample size is far too small to be truly statistically significant, but it's a nice start to start establishing early patterns. 

I would love to see what the distribution of most played Supports are across all ranks, but prepping that dataset for this kind of analysis would take several days due to my API call limits. And also probably create some gigantic files. 

**Question 2:** What is the win rate for the top 3 Support champions in this lobby? 


In [31]:
champion_pick_counts = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"), win_rate=("win", "mean")) #adds a column that returns win rate for each champion 
    .sort_values("pick_count", ascending=False)
)

champion_pick_counts.head(10)

,championName,pick_count,win_rate
40,Lux,75,0.573333
47,Morgana,66,0.621212
62,Seraphine,61,0.475410
83,Yuumi,48,0.520833
61,Senna,38,0.289474
35,Leona,34,0.558824
67,Sona,34,0.500000
55,Pyke,32,0.625000
49,Nautilus,32,0.437500
74,Thresh,30,0.566667


**Interpretation:**  
Even though Lux is the most popular Support pick (within this sample size), she doesn't have the highest win rate. I thought this was interesting because I was expecting Lux to have the highest win rate, but it just reminded me that just because a champion is the most popular pick doesn't mean they have the highest win rate. 

**Question 3:** Which Support champion has the highest win rate? 


In [ ]:
champion_pick_counts = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"), win_rate=("win", "mean")) #adds a column that returns win rate for each champion 
    .query("pick_count >= 30") #sorted the dataset by champions with a pick count of 30 or more because many champions only had 1-2 picks and 40 only returned like 4 champions
    .sort_values("win_rate", ascending=False)
)

champion_pick_counts.head(10)

,championName,pick_count,win_rate
55,Pyke,32,0.625000
47,Morgana,66,0.621212
40,Lux,75,0.573333
74,Thresh,30,0.566667
35,Leona,34,0.558824
83,Yuumi,48,0.520833
67,Sona,34,0.500000
62,Seraphine,61,0.475410
49,Nautilus,32,0.437500
61,Senna,38,0.289474


**Interpretation:**  
Even though Pyke is the 8th most popular Support pick, he has the highest win rate out of the top 10 most commonly played Support champions. Again, this sample size is still far too small to be truly statistically significant, but it's helpful to see some early patterns. As more data is added to the dataset, I'm expecting to see the difference in win rates decrease. 

It was also interesting for me to see that Senna has such a low win rate in this rank. As a player, I would probably avoid picking Senna for Ranked matches and learn how to play champions that counter Pyke, or at least learn more about how to play around him. 

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

#### Question 1: Most Popular Support Champion

In [ ]:
# Bar chart: Support pick counts from Question 1 (most → least)
champion_pick_chart = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"))
    .sort_values("pick_count", ascending=False)
)

#visualization components
fig = px.bar( 
    champion_pick_chart,
    x="championName", #x-axis
    y="pick_count", #y-axis 
    title="Lux is the most picked Support champion in Bronze IV ranked games", #these are pretty self-explanatory   
    labels={"championName": "Support champion", "pick_count": "Pick count"}, #labels for the axes
    category_orders={"championName": champion_pick_chart["championName"].tolist()}, #this is just to make sure the bars are in descending order
)
fig.show()


**Chart rationale:**  

I chose this chart type because I thought the height of each bar would best visually represent the difference in pick rates across each champion in the sample size. The reader should be able to determine who the most picked Support champions are for Bronze IV ranked lobbies at a quick glance.


#### Question 2: Win Rate for Top 3 Support Champions

In [45]:
# Horizontal bar chart: win rates for the top 3 most-picked Supports (Question 2)
top_3_win_rates = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"), win_rate=("win", "mean"))
    .sort_values("pick_count", ascending=False)
    .head(3)
    .sort_values("win_rate", ascending=True)
)
top_3_win_rates["win_rate_pct"] = top_3_win_rates["win_rate"] * 100

fig = px.bar(
    top_3_win_rates,
    x="win_rate_pct",
    y="championName",
    orientation="h",
    title="Morgana has the highest win rate among the top 3 most-picked Bronze IV Supports",
    labels={"championName": "Support champion", "win_rate_pct": "Win rate (%)"},
    category_orders={"championName": top_3_win_rates["championName"].tolist()[::-1]},
)
fig.update_xaxes(range=[0, 100])
fig.show()

**Chart rationale:**  

I chose this chart type because there's only three values I wanted to represent and the horizontal bar graph allows users to scan its contents quicker, in a natural list order. The reader should be able to see that within the Top 3 most commonly picked champions, Morgana has the highest win rate, not Lux (despite being the most picked champion). 


#### Question 3: Support Champion With the Highest Win Rate 

In [49]:
# Scatter plot: win rate vs. games played for Supports with 30+ picks (Question 3)
champion_win_rates = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"), win_rate=("win", "mean"))
    .query("pick_count >= 30")
    .assign(win_rate_pct=lambda d: d["win_rate"] * 100)
    .sort_values("win_rate", ascending=False)
)

fig = px.scatter(
    champion_win_rates,
    x="pick_count",
    y="win_rate_pct",
    text="championName",
    title="Higher win rates must be read alongside how many games each Support was played",
    labels={
        "pick_count": "Games played",
        "win_rate_pct": "Win rate (%)",
        "championName": "Support champion",
    },
    hover_data={"pick_count": True, "win_rate_pct": ":.1f", "championName": True},
)
fig.update_traces(textposition="top center", marker=dict(size=12))
fig.update_yaxes(range=[0, 100])
fig.show()

**Chart rationale:**  

I had initially selected another bar chart to represent and sort the champions with the highest win rate, but found that it didn't include information that tells the reader how many games were taken into consideration to calculate that win rate. As a result, I adjusted my visualization to be a scatter plot instead, where the user can now see which champion has the highest win rate (Pyke), but also have the full context of how these win rates were calculated and take that information with a grain of salt. 

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

This analysis definitely showed me what I was expecting, and even made me laugh because the top 3 support champions that I found from this analysis are also my personal top 3 champs to play (they're just really fun and have a lot of nice cosmetic skins). However, what surprised me was that the champions that are the most popular to play don't necessarily have the highest win rates. I first noticed this in Question 2, where I answered my question of "What is the win rate for the top 3 Support Champions?". The answer to this question piqued my interest in knowing which champion actually had the highest win rate overall, not just limited to the top 3 most popular Support champions. 

There are a couple questions I would like to explore next: 

4. What is the distribution for the most played Support champion across all Level IV ranks (ex: Silver IV, Platinum IV, etc.)? 
5. What is the most common Bottom Lane/Support pairing in Bronze-level lobbies? What about across all ranks? 

The limitations of this analysis mostly have to do with the API call limitations from Riot's end, in addition to the run times for the scripts that actually fetches the data from the API and storage limitations due to the massive amounts of data that gets returned. Additionally, all the data that's being used for this analysis is static, taken from a snapshot of my API calls from May 6th, 2026. So who knows- even though this initial analysis gave us a good look at initial patterns, they might change with the more data that gets included in the analysis. 